In [1]:
import pandas as pd
import numpy as np
import time 
from datetime import datetime
# Original DataFrame
data = [['c1', 2020,100, 500], ['c2', 2022,200, 300]]
df = pd.DataFrame(data, columns=['C', 'D', 'A', 'B'])

def generate_random(price_or, model: int, offset=0) -> int:
    """
    To generate the random data between a range 
    """
    np.random.seed(int(time.time()) % 100000 + offset)
    return np.random.randint(int(price_or*0.5), int(price_or*1.5))


# Function to generate new simulated data
def generate_simulated_data(df, num_new_rows=5, scale_factor=0.9):
    new_rows = []
    
    for _, row in df.iterrows():
        c, d,a_orig, b_orig = row['C'], row['D'], row['A'], row['B']
        
        for i in range(num_new_rows): 
            new_a = generate_random(a_orig, i) 
            
            # Scale B based on the new A value
            new_b = b_orig * (new_a / a_orig) * scale_factor  

            new_rows.append([c,d, new_a, int(new_b)])  # Convert B to integer

    # Create new DataFrame with simulated values
    df_new = pd.DataFrame(new_rows, columns=['C','D', 'A', 'B'])
    
    # Combine with original data
    return pd.concat([df, df_new], ignore_index=True)

# Generate simulated dataset
df_simulated = generate_simulated_data(df, num_new_rows=5)

# Display results
print(df_simulated)


     C     D    A    B
0   c1  2020  100  500
1   c2  2022  200  300
2   c1  2020   81  364
3   c1  2020   81  364
4   c1  2020   81  364
5   c1  2020   81  364
6   c1  2020   81  364
7   c2  2022  131  176
8   c2  2022  131  176
9   c2  2022  131  176
10  c2  2022  131  176
11  c2  2022  131  176


In [2]:


class Simulation_p_k():
    MEAN_KM : int = 15000
    KM_THRESHOLD: float = 0.85 
    MAX_PENALTY: float = 0.45
    def __init__(self, price: int, model: int, km: int):
        """ 
        Args: 
            price (int): Original price
            model (int): The model year of the vehicle 
            km (int) : The km of the vehicle
        """
        self.price = price 
        self.model = model
        self.km = km
    
    def _calculate_mean_km(self):
        """
        Calculate the mean km of a vehicle based on its model year and mean km.

        Args:
            self

        Returns:
            int: The calculated mean km.
        """
        date = datetime.now()
        year = date.year
        years_difference = year - int(self.model)
        
        if years_difference == 0:
            return self.km
        
        return years_difference * Simulation_p_k.MEAN_KM
    
    def simulate_km(self, offset=0) -> int:
        """
        To generate the random data between a range 

        Args:
            self
            offset (int) : An int to change the random seed
        
        Returns:
            int : new km value
        """
        average_km = self._calculate_mean_km()
        np.random.seed(int(time.time()) % 100000 + offset)
        return np.random.randint(int(average_km*0.5), int(average_km*1.5))
    
    def calculate_percentage(self, km: int) -> float:
        """
        Calculate the percentage of the price based on the vehicle's Simulation_p_k.

        Args:
            self

        Returns:
            float: The calculated percentage.
        """

        average_km = self._calculate_mean_km()
        if average_km*Simulation_p_k.KM_THRESHOLD < km <= average_km :
            return 1.0
        elif km <= average_km * Simulation_p_k.KM_THRESHOLD:
            return 1.03
        
        else:
            increments: float = 0.0
            for increment in range(1,15):
                increments += 0.03
                if km <= (average_km +(Simulation_p_k.MEAN_KM * increment)):
                    return round(1 - increments, 2)
                
            return round(1 - Simulation_p_k.MAX_PENALTY,2)
        
    def calculate_price(self, km: int) -> int:
        """
        Calculate the price of the vehicle based on the given price and percentage.

        Args:
            self

        Returns:
            float: The calculated price.
        """
        new_price: float = self.price * self.calculate_percentage(km)
        # new_price = self.price - discount_value
        return round(new_price)

    
        


In [14]:
data = [['fase1', 'day1','c1', 2020,15e3*5, 500], ['fase2', 'day2','c2', 2022,15e3*2, 300]]
df = pd.DataFrame(data, columns=['F', 'D','C', 'M', 'K', 'P'])

def generate_simulated_data(df, num_new_rows=5) -> pd.DataFrame:
    new_rows = []
    
    for _, row in df.iterrows():
        f, d, c, m, km_orig, pr_orig = row['F'], row['D'], row['C'], row['M'], row['K'], row['P']

        
        for i in range(num_new_rows): 
            simulation_obj = Simulation_p_k(pr_orig, m, km_orig)
            new_km = int(simulation_obj.simulate_km(i)) 
            # Scale B based on the new A value
            price_new = simulation_obj.calculate_price(new_km)

            new_rows.append([f,d,c,m, int(new_km), price_new])  # Convert B to integer

    # Create new DataFrame with simulated values
    df_new = pd.DataFrame(new_rows, columns=['F', 'D','C', 'M', 'K', 'P'])
    
    # Combine with original data
    return pd.concat([df, df_new], ignore_index=True)

# Generate simulated dataset
df_simulated = generate_simulated_data(df, num_new_rows=5)

# Display results
print(df_simulated)


        F     D   C     M        K    P
0   fase1  day1  c1  2020  75000.0  500
1   fase2  day2  c2  2022  30000.0  300
2   fase1  day1  c1  2020  58508.0  515
3   fase1  day1  c1  2020  82090.0  485
4   fase1  day1  c1  2020  81418.0  485
5   fase1  day1  c1  2020  91239.0  470
6   fase1  day1  c1  2020  61067.0  515
7   fase2  day2  c2  2022  35643.0  309
8   fase2  day2  c2  2022  67090.0  282
9   fase2  day2  c2  2022  66418.0  282
10  fase2  day2  c2  2022  62265.0  282
11  fase2  day2  c2  2022  46067.0  291


In [15]:
def clean_new_cars(df):
    df.loc[df['ESTADO_VENTA'] == 'Nuevos', 'KILOMETRAJE'] = 0
    return df.drop(columns=['ESTADO_VENTA'])

In [16]:
print(type(df_simulated))

<class 'pandas.core.frame.DataFrame'>
